# Домашняя работа

## Задание

Написать на питоне программу, которая извлекает из текста пары глагол+наречие (наречие+глагол):

бежать быстро, хорошо учиться и т.д.

1. Программа должна получать текст из файла a.txt, содержащего пару слов на каждой строке, которые нужно проверить на необходимое условие.

2. Программа должна импортировать только одну библиотеку - re

3. Программа не должна использовать каких-то словарей.

4. Программа должна реализовывать проверку по единственному паттерну регулярного выражения, длина паттерна не превышает 100байт.

## Датасет

In [4]:
def get_data_from_url(url: str) -> str | None:
    """Загрузить файл по ссылке"""
    import requests

    response = requests.get(url)
    if response.status_code == 200:
        data = response.text
        print("> Файл успешно загрузился!")
        return data
    else:
        print("> Произошла ошибка загрузки!")


text = get_data_from_url(
    "https://raw.githubusercontent.com/sanbob-vault/hw-ai/refs/heads/main/01-regex-verb-adverb/data/my-text.txt"
)
with open("my-text.txt", "w") as file:
    file.write(text or "")

> Файл успешно загрузился!


## Решение

In [5]:
import re
from pathlib import Path


def get_verb_regex() -> str:
    """Регулярное выражение для поиска глаголов"""

    """
    Глагол может заканчиваться на:
    (ет, ит, ут, ют, ат, ят + [е] ИЛИ ть) + [ся, сь]
    """

    return r"\w+([еиуюая]те?|ть)(с[яь])?\b"


def get_adverb_regex() -> str:
    """Регулярное выражение для поиска наречий"""

    """
    Наречие может заканчиваться на о, е, и, а, я, у
    """

    return r"\w+[оеаяу]\b"


def get_space_regex() -> str:
    """Регулярное выражение для нахождения общего контекста"""

    """
    Любые символы кроме точки и запятой
    Потому что между связанными наречием и глаголом могут быть другие слова, а точки и запятые редко
    """

    return r"[^.,]+?"


def get_combined_regex() -> str:
    """Регулярное выражения для поиска пар глаголов с наречиями"""

    verb = get_verb_regex()
    adverb = get_adverb_regex()
    space = get_space_regex()

    # (глагол + пространство + наречие) или (наречие + пространство + глагол)
    return rf"({verb}{space}{adverb}|{adverb}{space}{verb})"


def main(file_path: Path):
    pattern = get_combined_regex()

    print("-" * 32)
    print(
        f"Собранный паттерн: {pattern}\nКоличество символов в паттерне: {len(pattern)}"
    )
    print("-" * 32, "\n")

    try:
        with open(file_path, "r") as file:
            text = file.read()
    except FileNotFoundError:
        print(f"- Файл {file_path} не найден")
        return

    matches: list[str] = re.findall(pattern, text, flags=re.IGNORECASE)
    print("- Найденные пары:")
    print("Полная фраза -- Пара")
    for match in matches:
        full_phrase = match[0]
        first, last = full_phrase.split()[0], full_phrase.split()[-1]
        print(f"{full_phrase} -- {first} {last}")


if __name__ == "__main__":
    arg = Path("my-text.txt")
    main(arg)


--------------------------------
Собранный паттерн: (\w+([еиуюая]те?|ть)(с[яь])?\b[^.,]+?\w+[оеаяу]\b|\w+[оеаяу]\b[^.,]+?\w+([еиуюая]те?|ть)(с[яь])?\b)
Количество символов в паттерне: 99
-------------------------------- 

- Найденные пары:
Полная фраза -- Пара
Быстро бегать -- Быстро бегать
красиво удаляться -- красиво удаляться
Сегодня сильно дует -- Сегодня дует
Медленно облака плывут -- Медленно плывут
легко дует -- легко дует
солнце ярко светит -- солнце светит
громко поют -- громко поют
тихо шумит -- тихо шумит
Все кругом скоропостижно оживает -- Все оживает
красиво лежит -- красиво лежит
где восхитительно ручка валяется -- где валяется


Можно улучшить эти регулярные выражения:
- Улучшить захват глаголов и наречий, например, добавить минимальную длину для наречия
- Сделать, чтобы ненужные группы не захватывались (:?)
- Захватывать глагол и наречие отдельно, чтобы выражение отдавало готовую пару
- Добавить в символы разделители контекста(точка и запятая) восклицательный и вопросительный знаки
- Улучшить разделение контекста ограничением длины между глаголом и наречием

Но, тогда я бы перешел лимит в 100 байт

Конечное регулярное выражение ровно 99 байт в кодировке Windows-1251:

```
(\w+([еиуюая]те?|ть)(с[яь])?\b[^.,]+?\w+[оеаяу]\b|\w+[оеаяу]\b[^.,]+?\w+([еиуюая]те?|ть)(с[яь])?\b)
```